### Seiten zu INV/UNINV

In [ ]:
import re
import pandas as pd
from openpyxl import load_workbook

HEALTHY_XLSX = r"K:\Team\Böhmer_Michael\PAPER\Maestroni_Validierungsdaten.xlsx"
HEALTHY_SHEET = "Healthy"

INV_XLSX = r"K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_Maestroni_ML.xlsx"
INV_SHEET_READ = 0   # so liest du per Index …
# … aber zum Schreiben brauchen wir den NAMEN:
wb = load_workbook(INV_XLSX, read_only=True)
INV_SHEET_WRITE = wb.sheetnames[INV_SHEET_READ]   # z.B. "Tabelle1" o.ä.

def norm(s):
    s = str(s).strip().replace(" ", "_")
    return re.sub(r"__+", "_", s)

def split_lr(c):
    c = norm(c)
    m = re.search(r"_(R|L)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], m.group(1).upper()
    m = re.search(r"_(Right|Left)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], ('R' if m.group(1).lower()=='right' else 'L')
    return c, None

def split_inv(c):
    c = norm(c)
    m = re.search(r"_(INV|UNINV)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], m.group(1).upper()
    return c, None

# --- Einlesen ---
dfH = pd.read_excel(HEALTHY_XLSX, sheet_name=HEALTHY_SHEET, engine="openpyxl")
dfI = pd.read_excel(INV_XLSX, sheet_name=INV_SHEET_READ, engine="openpyxl")

# Normalisieren
dfH.columns = [norm(c) for c in dfH.columns]
dfI.columns = [norm(c) for c in dfI.columns]

# LR-Paare (Healthy) und INV/UNINV-Paare (INV)
H_pairs, I_pairs = {}, {}
for c in dfH.columns:
    b, lr = split_lr(c)
    if lr: H_pairs.setdefault(b, {})[lr] = c
for c in dfI.columns:
    b, tag = split_inv(c)
    if tag: I_pairs.setdefault(b, {})[tag] = c

# Nicht-seitige Spalten 1:1 kopieren (falls vorhanden)
for col in sorted(set(dfH.columns) & set(dfI.columns)):
    dfI[col] = dfH[col]

# Dominanz
DOM_COL = 'Limb_Dominance'  # wie von dir beschrieben
dom = dfH[DOM_COL].astype(str).str.strip().str.lower()

# Seitige Metriken mappen (dominant → INV, nicht-dominant → UNINV)
common = sorted(set(H_pairs) & set(I_pairs))
rows = min(len(dfH), len(dfI))

for b in common:
    srcL, srcR = H_pairs[b].get('L'), H_pairs[b].get('R')
    dstINV, dstUNINV = I_pairs[b].get('INV'), I_pairs[b].get('UNINV')
    if not (srcL and srcR and dstINV and dstUNINV):
        continue
    for i in range(rows):
        d = dom.iat[i]
        if d in ('left', 'l'):
            dfI.at[i, dstINV]   = dfH.at[i, srcL]
            dfI.at[i, dstUNINV] = dfH.at[i, srcR]
        elif d in ('right', 'r'):
            dfI.at[i, dstINV]   = dfH.at[i, srcR]
            dfI.at[i, dstUNINV] = dfH.at[i, srcL]

# --- Zurückschreiben: sheet_name MUSS String sein ---
with pd.ExcelWriter(INV_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    dfI.to_excel(xw, sheet_name=INV_SHEET_WRITE, index=False)


### Sortierung nach Header

In [ ]:
# -*- coding: utf-8 -*-
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import get_column_letter

FILE_PATH = r"K:\Team\Böhmer_Michael\PAPER\Maestroni_Validierungsdaten.xlsx"
SHEET_NAME = "ACLR"

# Umsortierbereich C–U, Block 3–35 (inkl. Ist-Header in Zeile 3)
COL_START = 3   # C
COL_END   = 21  # U
ROW_DATA_START = 3   # <-- Block beginnt in Zeile 3 (Ist-Header)
ROW_DATA_END   = 35

# Zielreihenfolge kommt aus Zeile 1
ROW_TARGET_ORDER = 1
NEW_SHEET_NAME   = "ACLR_reordered"

def normalize_label(s: Optional[str]) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    for ch in ["\n", "\r", "\t", "_", "-", "–", "—", "/", "\\", "|"]:
        s = s.replace(ch, " ")
    return " ".join(s.split()).lower()

def read_cell(ws: Worksheet, row: int, col: int):
    return ws.cell(row=row, column=col).value

def write_cell(ws: Worksheet, row: int, col: int, value):
    ws.cell(row=row, column=col, value=value)

def get_target_order(ws: Worksheet) -> Tuple[List[str], List[str]]:
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_TARGET_ORDER, c)
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def get_source_labels_from_row3(ws: Worksheet) -> Tuple[List[str], List[str]]:
    """Liest die Ist-Header aus Zeile 3 des Umsortierbereichs (C3..U3)."""
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_DATA_START, c)  # Zeile 3
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def build_mapping(src_norm: List[str], tgt_norm: List[str]):
    label_to_tgt = {}
    for i, lab in enumerate(tgt_norm):
        label_to_tgt.setdefault(lab, []).append(i)

    dup_targets = {lab: idxs for lab, idxs in label_to_tgt.items() if len(idxs) > 1 and lab != ""}

    colmap: Dict[int, int] = {}
    used_tgt = set()
    for s_idx, slab in enumerate(src_norm):
        if slab == "":
            continue
        for ti in label_to_tgt.get(slab, []):
            if ti not in used_tgt:
                colmap[s_idx] = ti
                used_tgt.add(ti)
                break

    missing_targets = [ti for ti in range(len(tgt_norm)) if ti not in used_tgt]
    extras_sources  = [si for si in range(len(src_norm)) if si not in colmap]
    return colmap, missing_targets, extras_sources, dup_targets

def buffer_blocks(ws: Worksheet, colmap: Dict[int, int]) -> Dict[int, List]:
    """Puffert den gesamten Block Zeile 3–35 der jeweils zugeordneten Quellspalte."""
    buf: Dict[int, List] = {}
    for s_idx, t_idx in colmap.items():
        abs_col = COL_START + s_idx
        data = [read_cell(ws, r, abs_col) for r in range(ROW_DATA_START, ROW_DATA_END + 1)]
        buf[t_idx] = data
    return buf

def create_or_replace_sheet(wb, name: str) -> Worksheet:
    if name in wb.sheetnames:
        wb.remove(wb[name])
    return wb.create_sheet(title=name)

def main():
    xlsx = Path(FILE_PATH)
    if not xlsx.exists():
        raise FileNotFoundError(f"File not found: {xlsx}")
    wb = load_workbook(xlsx)
    if SHEET_NAME not in wb.sheetnames:
        raise ValueError(f"Sheet '{SHEET_NAME}' not found.")
    ws_src: Worksheet = wb[SHEET_NAME]

    # Ziel (Zeile 1) und Ist (Zeile 3) lesen
    tgt_raw, tgt_norm = get_target_order(ws_src)
    src_raw, src_norm = get_source_labels_from_row3(ws_src)

    colmap, missing_targets, extras_sources, dup_targets = build_mapping(src_norm, tgt_norm)

    # Phase 1: Puffer
    buffer = buffer_blocks(ws_src, colmap)

    # Phase 2: Schreiben in neues Blatt
    ws_out = create_or_replace_sheet(wb, NEW_SHEET_NAME)

    # (Optional) Linke Spalten A..B kopieren (Zeilen 1..ROW_DATA_END)
    for c in range(1, COL_START):
        for r in range(1, ROW_DATA_END + 1):
            write_cell(ws_out, r, c, read_cell(ws_src, r, c))

    # Zeile 1 (Ziel-Header) setzen
    for i, raw_label in enumerate(tgt_raw):
        write_cell(ws_out, ROW_TARGET_ORDER, COL_START + i, raw_label)

    # Zeile 2 bewusst leer lassen (keine separaten Header schreiben)
    for i in range(COL_END - COL_START + 1):
        write_cell(ws_out, 2, COL_START + i, None)

    # Block Zeilen 3–35 gemäß Mapping schreiben
    for t_idx in range(COL_END - COL_START + 1):
        abs_col_out = COL_START + t_idx
        if t_idx in buffer:
            data = buffer[t_idx]
            for off, val in enumerate(data):
                write_cell(ws_out, ROW_DATA_START + off, abs_col_out, val)
        else:
            # fehlende Zielspalte leer lassen
            for r in range(ROW_DATA_START, ROW_DATA_END + 1):
                write_cell(ws_out, r, abs_col_out, None)

    # Extras rechts anhängen (optional)
    if extras_sources:
        out_c = COL_END + 1
        for s_idx in extras_sources:
            abs_col_src = COL_START + s_idx
            # kopiere auch Zeilen 1..35 für Transparenz
            for r in range(1, ROW_DATA_END + 1):
                write_cell(ws_out, r, out_c, read_cell(ws_src, r, abs_col_src))
            out_c += 1

    # Report
    print("=== Reorder Summary ===")
    print(f"Source sheet  : {SHEET_NAME}")
    print(f"Output sheet  : {NEW_SHEET_NAME}")
    print(f"Block moved   : {get_column_letter(COL_START)}{ROW_DATA_START}–{get_column_letter(COL_END)}{ROW_DATA_END}")
    print(f"Mapped columns: {len(colmap)} / {COL_END - COL_START + 1}")
    if dup_targets:
        print("Duplicate target labels (normalized):")
        for lab, idxs in dup_targets.items():
            cols = [get_column_letter(COL_START + i) for i in idxs]
            print(f"  '{lab}' at targets {cols}")
    if missing_targets:
        cols = [get_column_letter(COL_START + i) for i in missing_targets]
        names = [tgt_raw[i] for i in missing_targets]
        print(f"Missing targets: {list(zip(cols, names))}")
    if extras_sources:
        cols = [get_column_letter(COL_START + i) for i in extras_sources]
        names = [src_raw[i] for i in extras_sources]
        print(f"Extra sources : {list(zip(cols, names))}")

    wb.save(xlsx)
    print(f"Saved: {xlsx}")
    print("Done.")

if __name__ == "__main__":
    main()
